# Camera-ready: LayerNorm rerun (reviewer xTi3)

Reruns the LayerNorm experiment with a corrected gradient of the sharpness. Autograd through `nn.LayerNorm` gave wrong
third-order derivatives, so `c_y` and `alpha` of the earlier LayerNorm runs are wrong (sharpness and loss are fine).
The LayerNorm in `src/archs.py` is now written out by hand; cell 4 checks the gradient against finite differences.

## How to run it
1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. **Run cells 1–3 (setup).** Cell 3 copies the finished unnormalized-MLP run from `MyDrive/slt_camera_ready`,
   so it is not run again. Results go to the new folder `MyDrive/slt_camera_ready_lnfix`.
3. **Run cell 4 (checks, about 5 min).** It must print **`GRAD CHECK PASSED`** and **`SMOKE TEST PASSED`**.
   If not, stop and send the output.
4. **Run cell 5 and leave it running.** After a disconnect, run cells 1–3 and then cell 5 again; finished runs are skipped.
5. **Cell 6** shows progress at any time. **Cell 7** writes the CSVs and plots once runs have finished.
   Send the folder `MyDrive/slt_camera_ready_lnfix/camera_ready_out/`.

## What runs (in this order), η = 0.02, 10k steps each, about 70 min per run on a T4
| Run | Purpose |
|---|---|
| hidden-LayerNorm MLP, γ = 0.05 | the rebuttal setting: no collapse, `c_y` vs `c_y^crit`, decomposition |
| hidden-LayerNorm MLP, γ = 0 | sharpness without weight decay |
| hidden-LayerNorm MLP, γ = 0.02 | intermediate γ |

The first run is the important one: after it finishes (about 70 min) cell 7 can already be run.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code

In [ ]:
import os
if not os.path.isdir('/content/slt-experiments'):
    !git clone -b camera-ready https://github.com/lutscha/slt-experiments.git /content/slt-experiments
else:
    !cd /content/slt-experiments && git pull
%cd /content/slt-experiments
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Paths, and copy the finished unnormalized-MLP run

In [ ]:
import os, shutil
os.environ['DATASETS'] = '/content/datasets'
os.environ['RESULTS'] = '/content/drive/MyDrive/slt_camera_ready_lnfix'
os.makedirs(os.environ['DATASETS'], exist_ok=True)
os.makedirs(os.environ['RESULTS'], exist_ok=True)

run_dir = 'cifar10-5k/full_200/seed_0/mse/gd/lr_0.02/wd_0.05'
old, new = f'/content/drive/MyDrive/slt_camera_ready/{run_dir}', f"{os.environ['RESULTS']}/{run_dir}"
if os.path.isdir(new):
    print('unnormalized run already in place')
elif os.path.isdir(old):
    shutil.copytree(old, new)
    print('copied the unnormalized run from', old)
else:
    print('unnormalized run not found; cell 5 will run it again (+70 min)')

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
print('Results will be saved to', os.environ['RESULTS'])

## 4. Checks (must print `GRAD CHECK PASSED` and `SMOKE TEST PASSED`)
The gradient check compares `c_y` with finite differences of the sharpness. The smoke test runs both networks for
a few steps on a 1k-sample subset; its results go to `/content/smoke`, not Drive.

In [ ]:
import os, shutil, subprocess, time
env = dict(os.environ, RESULTS='/content/smoke')
shutil.rmtree('/content/smoke', ignore_errors=True)
commands = [
    'camera_ready/grad_check.py',
    'camera_ready/run.py layernorm --smoke --only 0',
    'camera_ready/run.py layernorm --smoke --only 1',
    'camera_ready/analyze.py --smoke',
    'camera_ready/smoke_check.py',
]
for c in commands:
    t = time.time()
    p = subprocess.run('python ' + c, shell=True, env=env, capture_output=True, text=True)
    print(f'[{time.time() - t:6.1f}s] {c}')
    if p.returncode != 0 or c.endswith(('smoke_check.py', 'grad_check.py')):
        print(p.stdout[-4000:]); print(p.stderr[-4000:])
    if p.returncode != 0:
        raise RuntimeError(f'Failed: {c}  (send the output above)')

## 5. LayerNorm runs (about 3.5 h)
The log goes to `MyDrive/slt_camera_ready_lnfix/log_layernorm.txt`.

In [ ]:
import os, subprocess, time
t = time.time()
log = f"{os.environ['RESULTS']}/log_layernorm.txt"
print(f'=== layernorm started {time.strftime("%H:%M:%S")}, log: {log}', flush=True)
with open(log, 'a') as f:
    p = subprocess.run(['python', 'camera_ready/run.py', 'layernorm'], stdout=f, stderr=subprocess.STDOUT)
print(f"=== layernorm {'OK' if p.returncode == 0 else f'FAILED (exit {p.returncode}), see log'} "
      f'after {(time.time() - t) / 60:.0f} min', flush=True)
!python camera_ready/analyze.py
!python camera_ready/status.py

## 6. Progress (run any time)

In [ ]:
!python camera_ready/status.py 2>/dev/null | head -5
!tail -n 3 $RESULTS/log_layernorm.txt

## 7. Analysis: CSVs and quick-look plot in `MyDrive/slt_camera_ready_lnfix/camera_ready_out/`

In [ ]:
!python camera_ready/analyze.py
from IPython.display import Image, display
display(Image(os.environ['RESULTS'] + '/camera_ready_out/layernorm.png'))